# Few-Shot NER (5-shot) — ProcTemIST con GPT-4.1-mini

## Contenido

1. [Instalación](#instalación)
2. [Importaciones](#importaciones)
3. [Configuración](#configuración)
4. [Preparación](#preparación)
5. [Función de inferencia](#función-de-inferencia)
6. [Inferencia](#inferencia)
7. [Evaluación](#evaluación)


## Instalación

In [1]:
!pip install openai pandas tiktoken spacy --quiet
!python -m spacy download es_core_news_md --quiet


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 27.7 MB/s eta 0:00:0000:0100:01
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Importaciones

In [2]:
import os
import json
import time
import re
from pathlib import Path

import pandas as pd
import spacy
from openai import OpenAI
import warnings

warnings.filterwarnings('ignore')


## Configuración

In [ ]:
# ── API ────────────────────────────────────────────────────────────────────
try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret('OPENAI_API_KEY')
except Exception:
    api_key = os.environ.get('OPENAI_API_KEY')

client = OpenAI(api_key=api_key)

# ── Modelo ──────────────────────────────────────────────────────────────────
MODEL_NAME = 'gpt-4.1-mini-2025-04-14'

# ── spaCy ────────────────────────────────────────────────────────────────────
SPACY_MODEL    = 'es_core_news_md'
MAX_CHUNK_CHARS = None  # None = una oración por chunk

# ── Rutas ProcTemIST ──────────────────────────────────────────────────────────
PROJECT_ROOT         = "/kaggle/input/datasets/user"
PROCTEMIST_ROOT      = f"{PROJECT_ROOT}/proctemist"
TEXT_FILES_TEST_DIR  = f"{PROCTEMIST_ROOT}/text_files_test"
GS_TEST_TSV          = f"{PROCTEMIST_ROOT}/medprocner_tsv_test_subtask1.tsv"
PREDICTIONS_TSV      = "proctemist_openai_predictions.tsv"
EVAL_SUMMARY_JSON    = "evaluation_summary_proctemist.json"

MAX_TEST_FILES   = 250   # None para procesar todos
PRINT_RAW_OUTPUT = True

# ── Prompt del sistema ────────────────────────────────────────────────────────
SYSTEM_PROMPT = (
    "Actua como un sistema NER medico de alta precision.\n\n"
    "REGLAS DE EXTRACCION:\n"
    "1. Extrae exclusivamente entidades de tipo PROCEDIMIENTO medico (exploraciones, pruebas diagnosticas, intervenciones, tratamientos, cirugias, etc.).\n"
    "2. COPIA Y PEGA de forma literal: No cambies mayusculas, minusculas ni tildes.\n"
    "3. PROHIBIDO USAR SINONIMOS: Si el texto dice 'radiografia', no escribas 'Rx'.\n"
    "4. REPETICIONES: Si un procedimiento aparece varias veces en el texto, debes listarlo varias veces en lineas separadas.\n"
    "5. ORDEN: Extrae las menciones en el mismo orden en que aparecen en el texto.\n"
    "6. FORMATO: Únicamente el texto de la mención, una por línea. PROHIBIDO usar caracteres de lista al inicio.\n"
    "7. Si no hay nada, devuelve un texto vacio."
)

print(f'Modelo  : {MODEL_NAME}')
print(f'API OK  : {client is not None}')
print(f'MAX_TEST_FILES   : {MAX_TEST_FILES}')
print(f'PRINT_RAW_OUTPUT : {PRINT_RAW_OUTPUT}')


Modelo  : gpt-4.1-mini-2025-04-14
API OK  : True
MAX_TEST_FILES   : 250
PRINT_RAW_OUTPUT : True


## Preparación

In [6]:
nlp_spacy = spacy.load(SPACY_MODEL, disable=['ner', 'lemmatizer'])
print(f'spaCy: {nlp_spacy.meta["name"]} v{nlp_spacy.meta["version"]}')


def chunk_text_by_sentences(text, nlp, max_chars=MAX_CHUNK_CHARS):
    """Divide el texto en chunks de oraciones completas con sus offsets.
    Si max_chars es None, devuelve una oración por chunk."""
    doc = nlp(text)
    if max_chars is None:
        return [(text[s.start_char:s.end_char], s.start_char, s.end_char) for s in doc.sents]

    chunks, current_chars, chunk_start, current_end = [], 0, None, None
    for sent in doc.sents:
        if current_chars + len(sent.text) > max_chars and chunk_start is not None:
            chunks.append((text[chunk_start:current_end], chunk_start, current_end))
            current_chars, chunk_start, current_end = 0, None, None
        if chunk_start is None:
            chunk_start = sent.start_char
        current_chars += len(sent.text)
        current_end = sent.end_char
    if chunk_start is not None:
        chunks.append((text[chunk_start:current_end], chunk_start, current_end))
    return chunks


txt_files_test = list(Path(TEXT_FILES_TEST_DIR).glob('*.txt'))
print(f'Archivos de test: {len(txt_files_test)}')


spaCy: core_news_md v3.8.0
Archivos de test: 250


## Función de inferencia

In [7]:
# ── Ejemplos few-shot (5-shot) ──────────────────────────────────────────────
EXAMPLES = [
    {
        "text": "En la exploración física destaca palidez de piel, Tª: 37,2 ºC y en la auscultación pulmonar crepitantes en base derecha.",
        "mentions": "exploración física\nauscultación pulmonar",
    },
    {
        "text": "Paciente de 38 años, con amputación de antepié izquierdo por arteriopatía por enfermedad de Buerger.",
        "mentions": "amputación de antepié izquierdo",
    },
    {
        "text": "Dada la edad de la enferma y con el diagnóstico de neoplasia de cabeza de páncreas, se optó por tratamiento paliativo con colocación de endoprótesis (endoprótesis biliar tipo wall stent de 10 x 70 mm de longitud) quedando alojada desde colédoco proximal hasta el interior de la luz duodenal.",
        "mentions": "tratamiento paliativo\ncolocación de endoprótesis\nendoprótesis biliar tipo wall stent",
    },
    {
        "text": "La paciente se mantuvo en DPA hasta agosto de 2002, momento en el que se le realiza un trasplante renal, precisando trasplantectomía por trombosis.",
        "mentions": "DPA\ntrasplante renal\ntrasplantectomía",
    },
    {
        "text": "Los padres activaban el aparato en casa para lograr la expansión y una vez por semana nosotros modificábamos la tensión de las cadenetas para lograr la retrusión de la premaxila.",
        "mentions": "expansión\nmodificábamos la tensión de las cadenetas\nretrusión de la premaxila",
    },
]


def build_messages(chunk_text):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT}]
    for ex in EXAMPLES:
        messages.append({'role': 'user',      'content': f"Texto para analizar:\n{ex['text']}"})
        messages.append({'role': 'assistant', 'content': ex['mentions']})
    messages.append({'role': 'user', 'content': f'Texto para analizar:\n{chunk_text}'})
    return messages


def call_openai_inference(chunk_text):
    response = client.chat.completions.create(
        model       = MODEL_NAME,
        messages    = build_messages(chunk_text),
        temperature = 0,
        max_tokens  = 300,
    )
    return response.choices[0].message.content.strip()


print(f'Función de inferencia lista (5-shot). Modelo: {MODEL_NAME}')


Función de inferencia lista (5-shot). Modelo: gpt-4.1-mini-2025-04-14


## Inferencia

In [8]:
pred_rows       = []
inference_times = []
test_files = sorted(txt_files_test)
if MAX_TEST_FILES is not None:
    test_files = test_files[:MAX_TEST_FILES]
total_files = len(test_files)
print(f'Archivos de test a procesar : {total_files}')
print(f'Modelo de inferencia        : {MODEL_NAME}\n')
for i, txt_path in enumerate(test_files):
    text         = txt_path.read_text(encoding='utf-8')
    raw_mentions = []
    t0           = time.time()
    chunks = chunk_text_by_sentences(text, nlp_spacy)
    for chunk_idx, (chunk_text, chunk_char_start, chunk_char_end) in enumerate(chunks):
        try:
            result = call_openai_inference(chunk_text)
        except Exception as e:
            print(f'  Error en inferencia: {e}')
            continue
        if PRINT_RAW_OUTPUT:
            print(f"  [chunk {chunk_idx+1}/{len(chunks)} | chars {chunk_char_start}-{chunk_char_end}]")
            print(f"  RAW: {repr(result)}")
        search_cursors = {}
        for mention in [line.strip() for line in result.splitlines() if line.strip()]:
            pattern = re.escape(mention)
            cursor  = search_cursors.get(mention, chunk_char_start)
            match = re.search(pattern, text[cursor:chunk_char_end], re.IGNORECASE)
            if not match:
                match = re.search(pattern, text[chunk_char_start:chunk_char_end], re.IGNORECASE)
                if not match:
                    continue
                off0 = chunk_char_start + match.start()
                off1 = chunk_char_start + match.end()
            else:
                off0 = cursor + match.start()
                off1 = cursor + match.end()
            search_cursors[mention] = off1
            raw_mentions.append((text[off0:off1], off0, off1))
    t_infer = time.time() - t0
    inference_times.append(t_infer)
    raw_mentions.sort(key=lambda x: x[1])
    seen_offsets = set()
    deduped      = []
    for span_text, off0, off1 in raw_mentions:
        if (off0, off1) not in seen_offsets:
            seen_offsets.add((off0, off1))
            deduped.append((span_text, off0, off1))
    for mark_idx, (span_text, off0, off1) in enumerate(deduped, 1):
        pred_rows.append({
            "filename":   txt_path.stem,
            "ann_id":     f"T{mark_idx}",
            "label":      "PROCEDIMIENTO",
            "start_span": off0,
            "end_span":   off1,
            "text":       span_text,
        })
    print(f'[{i+1}/{total_files}] {txt_path.name} | {t_infer:.2f}s | {len(chunks)} chunks | {len(deduped)} menciones')
pd.DataFrame(pred_rows, columns=["filename", "ann_id", "label", "start_span", "end_span", "text"]).to_csv(
    PREDICTIONS_TSV, sep="\t", index=False
)
total_time = sum(inference_times)
print(f'\nPredicciones guardadas en {PREDICTIONS_TSV}')
print(f'Tiempo total        : {total_time:.2f}s  ({total_time / 60:.2f}m)')
if inference_times:
    print(f'Tiempo medio/archivo: {total_time / len(inference_times):.2f}s')


Archivos de test a procesar : 250
Modelo de inferencia        : gpt-4.1-mini-2025-04-14

  [chunk 1/13 | chars 0-90]
  RAW: ''
  [chunk 2/13 | chars 91-170]
  RAW: ''
  [chunk 3/13 | chars 171-204]
  RAW: 'exploración física'
  [chunk 4/13 | chars 205-347]
  RAW: 'ultrasonido abdominal'
  [chunk 5/13 | chars 348-472]
  RAW: 'Urograma descendente'
  [chunk 6/13 | chars 473-667]
  RAW: 'TAC helicoidal'
  [chunk 7/13 | chars 668-744]
  RAW: 'estudio hormonal para la determinación de cortisol'
  [chunk 8/13 | chars 744-799]
  RAW: 'cirugía por abordaje lumbar'
  [chunk 9/13 | chars 800-949]
  RAW: ''
  [chunk 10/13 | chars 950-1003]
  RAW: 'adrenalectomía derecha'
  [chunk 11/13 | chars 1004-1090]
  RAW: ''
  [chunk 12/13 | chars 1090-1294]
  RAW: ''
  [chunk 13/13 | chars 1295-1397]
  RAW: 'examen histopatológico'
[1/250] S0004-06142006000100010-1.txt | 12.04s | 13 chunks | 8 menciones
  [chunk 1/21 | chars 0-197]
  RAW: 'gastrectomía subtotal'
  [chunk 2/21 | chars 198-412]
  RAW: ''
  [

## Evaluación

In [9]:
import json
import pandas as pd


def prf(tp, fp, fn):
    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * p * r / (p + r) if (p + r) else 0.0
    return p, r, f1

df_pred = pd.read_csv(PREDICTIONS_TSV, sep='\t')
df_gs = pd.read_csv(GS_TEST_TSV, sep='\t')
df_gs = df_gs[df_gs['filename'].isin(df_pred['filename'].unique())]

set_gs = set(zip(df_gs['filename'], df_gs['label'], df_gs['start_span'], df_gs['end_span']))
set_pred = set(zip(df_pred['filename'], df_pred['label'], df_pred['start_span'], df_pred['end_span']))

tp, fp, fn = len(set_gs & set_pred), len(set_pred - set_gs), len(set_gs - set_pred)
p, r, f1 = prf(tp, fp, fn)

report = {'Estricta': {'tp': tp, 'fp': fp, 'fn': fn,
                       'precision': round(p, 4), 'recall': round(r, 4), 'fscore': round(f1, 4)}}

print(f'── ESTRICTA ── P={p:.4f} R={r:.4f} F1={f1:.4f} (TP={tp} FP={fp} FN={fn})\n')

thresholds = [0.0, 0.5, 0.8]
results = {t: {'tp': 0, 'fp': 0, 'fn': 0} for t in thresholds}

for filename in df_pred['filename'].unique():
    gs_ints = list(zip(df_gs[df_gs['filename'] == filename]['start_span'],
                       df_gs[df_gs['filename'] == filename]['end_span']))
    pred_ints = list(zip(df_pred[df_pred['filename'] == filename]['start_span'],
                         df_pred[df_pred['filename'] == filename]['end_span']))
    iou_matrix = sorted(
        [(max(0, min(p1,g1) - max(p0,g0)) / (max(p1,g1) - min(p0,g0)), pi, gi)
         for pi, (p0, p1) in enumerate(pred_ints)
         for gi, (g0, g1) in enumerate(gs_ints)
         if max(p1,g1) - min(p0,g0) > 0 and min(p1,g1) - max(p0,g0) > 0],
        reverse=True,
    )
    for t in thresholds:
        matched_p, matched_g = set(), set()
        for iou, pi, gi in iou_matrix:
            if iou >= t and pi not in matched_p and gi not in matched_g:
                matched_p.add(pi)
                matched_g.add(gi)
        tp_t = len(matched_p)
        results[t]["tp"] += tp_t
        results[t]["fp"] += len(pred_ints) - tp_t
        results[t]["fn"] += len(gs_ints)   - tp_t

print("── IoU ──")
for t in thresholds:
    tp_t, fp_t, fn_t = results[t]["tp"], results[t]["fp"], results[t]["fn"]
    p_t, r_t, f1_t   = prf(tp_t, fp_t, fn_t)
    report[f"IoU >= {t}"] = {"tp": tp_t, "fp": fp_t, "fn": fn_t,
                              "precision": round(p_t, 4), "recall": round(r_t, 4), "fscore": round(f1_t, 4)}
    print(f"IoU >= {t}:  P={p_t:.4f}  R={r_t:.4f}  F1={f1_t:.4f}  (TP={tp_t} FP={fp_t} FN={fn_t})")

with open(EVAL_SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

── ESTRICTA ── P=0.5891 R=0.5437 F1=0.5655 (TP=1967 FP=1372 FN=1651)

── IoU ──
IoU >= 0.0:  P=0.7940  R=0.7325  F1=0.7620  (TP=2651 FP=688 FN=968)
IoU >= 0.5:  P=0.6861  R=0.6330  F1=0.6585  (TP=2291 FP=1048 FN=1328)
IoU >= 0.8:  P=0.6101  R=0.5629  F1=0.5855  (TP=2037 FP=1302 FN=1582)
